In [2]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
# !pip install numpy pandas matplotlib seaborn -q

import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False   # 마이너스 부호 깨짐 방지
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3


In [3]:
# ─────────────────────────────────────────────
# 종합 실습용 데이터 — 새 스냅샷 (이 셀부터 단독 실행 가능)
# ─────────────────────────────────────────────
np.random.seed(7)

n_cust = 200
cust = pd.DataFrame({
    "customer_id": [f"C{str(i).zfill(4)}" for i in range(1, n_cust + 1)],
    "region": np.random.choice(["서울", "경기", "부산", "인천"], n_cust),
    "membership": np.random.choice(["basic", "premium", "vip"], n_cust, p=[0.6, 0.3, 0.1]),
})

cats = ["패션", "뷰티", "식품", "가전"]
n_prod = 30
prod = pd.DataFrame({
    "product_id": [f"P{str(i).zfill(3)}" for i in range(1, n_prod + 1)],
    "category": np.random.choice(cats, n_prod),
    "price": np.random.choice([12000, 25000, 40000, 75000], n_prod),
})

n_ord = 1500
oc = np.random.choice(cust["customer_id"], n_ord)
op = np.random.choice(prod["product_id"], n_ord)
qty = np.random.choice([1, 1, 2, 3], n_ord)
amt = prod.set_index("product_id")["price"].loc[op].values * qty
odate = pd.to_datetime("2025-01-01") + pd.to_timedelta(np.random.randint(0, 150, n_ord), unit="D")
ordr = pd.DataFrame({
    "order_id": [f"O{str(i).zfill(5)}" for i in range(1, n_ord + 1)],
    "customer_id": oc, "product_id": op,
    "quantity": qty, "amount": amt.astype(float), "order_date": odate,
})
print("스냅샷 준비 완료 — orders:", ordr.shape, "| customers:", cust.shape, "| products:", prod.shape)

스냅샷 준비 완료 — orders: (1500, 6) | customers: (200, 3) | products: (30, 3)


In [ ]:
print("병합 전 세 검증")
print("customers 키 중복 :", cust["customer_id"].duplicated().sum(), "건")
print("products 키 중복 : ", prod["product_id"].duplicated().sum(), "건")
print("매칭 안 되는 customer_id:", (~ordr["customer_id"].isin(cust["customer_id"])).sum(), "건")
print("매칭 안 되는 product_id :", (~ordr["product_id"].isin(prod["product_id"])).sum(), "건")

mergedf = (
    ordr.merge(cust, on="customer_id", how="left", validate="m:1")
    .merge(prod, on="product_id", how="left", validate="m:1")
)
# 한 고객 당 다양한 주문을 하기 떄문에 m(ordr):1(cust) 관계 정립
# 여러 주문이 한 상품(상품 정보는 한 번만 존재)을 구매하기 때문에 m(ordr):1(cust) 관계 정립

print(mergedf.head())

병합 전 세 검증
customers 키 중복 : 0 건
products 키 중복 :  0 건
매칭 안 되는 customer_id: 0 건
매칭 안 되는 product_id : 0 건
  order_id customer_id product_id  quantity    amount order_date region  \
0   O00001       C0032       P005         3  120000.0 2025-04-12     서울   
1   O00002       C0049       P020         1   25000.0 2025-03-09     서울   
2   O00003       C0041       P016         3   36000.0 2025-01-27     인천   
3   O00004       C0123       P002         3  120000.0 2025-03-26     경기   
4   O00005       C0150       P014         3  120000.0 2025-03-16     경기   

  membership category  price  
0    premium       패션  40000  
1      basic       식품  25000  
2      basic       패션  12000  
3      basic       식품  40000  
4      basic       가전  40000  


In [5]:
region_membership = pd.pivot_table(
    mergedf, index="region", columns="membership", values="amount",
    aggfunc="sum", fill_value=0, margins=True 
)

print("[지역 × 회원등급 매출 교차표] (pitot_table)") 
print(region_membership.head())

summary_cate = (
    mergedf.groupby("category")
    .agg(
        매출=("amount", "sum"),
        건수=("order_id", "count"),
        객단가=("amount", "mean")
    )
)

print("[카테고리별 KPI(매출·건수·객단가) 요약표] (groupby + agg)]")
print(summary_cate.head())

[지역 × 회원등급 매출 교차표] (pitot_table)
membership       basic     premium        vip         All
region                                                   
경기          14594000.0   3226000.0  3325000.0  21145000.0
부산          11531000.0   4857000.0  2366000.0  18754000.0
서울          11914000.0   9244000.0  2322000.0  23480000.0
인천          17372000.0   9675000.0   530000.0  27577000.0
All         55411000.0  27002000.0  8543000.0  90956000.0
[카테고리별 KPI(매출·건수·객단가) 요약표] (groupby + agg)]
                  매출   건수           객단가
category                               
가전        20248000.0  398  50874.371859
뷰티         9986000.0  192  52010.416667
식품        22404000.0  398  56291.457286
패션        38318000.0  512  74839.843750


In [6]:
mergedf["region_ratio"] = (
    mergedf["amount"] /
    mergedf.groupby("region")["amount"].transform("sum")
)

print(mergedf.head())

  order_id customer_id product_id  quantity    amount order_date region  \
0   O00001       C0032       P005         3  120000.0 2025-04-12     서울   
1   O00002       C0049       P020         1   25000.0 2025-03-09     서울   
2   O00003       C0041       P016         3   36000.0 2025-01-27     인천   
3   O00004       C0123       P002         3  120000.0 2025-03-26     경기   
4   O00005       C0150       P014         3  120000.0 2025-03-16     경기   

  membership category  price  region_ratio  
0    premium       패션  40000      0.005111  
1      basic       식품  25000      0.001065  
2      basic       패션  12000      0.001305  
3      basic       식품  40000      0.005675  
4      basic       가전  40000      0.005675  


# 보고서
---
## 1. 분석 개요
- 대상: 모두마켓 주문 1,500건 (2025-01 ~ 2025-05), 고객·상품 정보 병합
- 방법: 3개 테이블 키 검증 후 m:1 병합 (자세한 근거는 코드에 작성함) → 카테고리별 KPI 집계 → 교차표·각 주문별 지역 매출 점유율 분석

## 2. 발견 3가지
- 매출이 가장 높았던 카테고리는 38318000원으로 패션이었으며, 주문 건수도 512건으로 카테고리중에서 가장 높은 수치를 보임
- 지역 중 인천 지역의 매출 기여가 가장 높았음
- Basic, Premium 별 매출도 인천이 가장 높았으나, VIP 메출은 현저히 떨어짐

## 3. 데이터 신뢰성 (반드시 기록)
- 병합 전 키 검증: 중복 0건, 미매칭 0건 → 행 폭증·조용한 결측 없음.
- validate="m:1" 통과 → 관계 가정(주문:상품 = m:1) 유효.

## 4. 제언
- 인천 지역 VIP 매출의 원인을 파악하고 보이며 이에 맞는 프로모션 개발이 필요해 보임